In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, accuracy
from sklearn.metrics.pairwise import cosine_similarity

ratings=pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/ratings.csv")
movies =pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/movies.csv")
tags   =pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/tags.csv")

ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
split_index = int(len(ratings_sorted) * 0.8)

train_df = ratings_sorted.iloc[:split_index]
test_df = ratings_sorted.iloc[split_index:]

print("Train:", train_df.shape, "Test:", test_df.shape)

Train: (80668, 4) Test: (20168, 4)


In [2]:
# How many ratings does each user have in the training set?
user_rating_counts = train_df.groupby('userId').size()

print("Ratings per user in train — min:", user_rating_counts.min(), 
      ", max:", user_rating_counts.max(), 
      ", mean:", user_rating_counts.mean())
print("\nDistribution:")
print(user_rating_counts.describe())

Ratings per user in train — min: 16 , max: 2516 , mean: 154.53639846743295

Distribution:
count     522.000000
mean      154.536398
std       247.402723
min        16.000000
25%        35.000000
50%        67.500000
75%       159.750000
max      2516.000000
dtype: float64


In [3]:
# Check how many TEST users fall below a threshold in TRAIN (candidates for cold-start fallback)
threshold = 5  # a common convention, we can adjust based on what we see above

test_users_list = test_df['userId'].unique()
cold_start_users = [u for u in test_users_list if user_rating_counts.get(u, 0) < threshold]

print(f"Test users below threshold ({threshold} ratings in train): {len(cold_start_users)} out of {len(test_users_list)}")
print(f"That's {len(cold_start_users)/len(test_users_list):.2%} of test users who are 'cold-start' cases")

Test users below threshold (5 ratings in train): 88 out of 116
That's 75.86% of test users who are 'cold-start' cases


In [4]:
# How many test users have ZERO ratings in train (fully new) vs some ratings below threshold
zero_in_train = [u for u in test_users_list if user_rating_counts.get(u, 0) == 0]
print(f"Test users with ZERO training history: {len(zero_in_train)} out of {len(test_users_list)}")

Test users with ZERO training history: 88 out of 116


In [5]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, accuracy
from sklearn.metrics.pairwise import cosine_similarity

# Load data
ratings=pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/ratings.csv")
movies =pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/movies.csv")
# Time-based split (same as always)
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
split_index = int(len(ratings_sorted) * 0.8)
train_df = ratings_sorted.iloc[:split_index]
test_df = ratings_sorted.iloc[split_index:]

print("Train:", train_df.shape, "Test:", test_df.shape)

Train: (80668, 4) Test: (20168, 4)


In [7]:
reader = Reader(rating_scale=(0.5, 5.0))
train_data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)
trainset = train_data.build_full_trainset()

# Using your tuned best params from Phase 4 - replace with your actual gs.best_params['rmse'] values
svd_model = SVD(n_factors=100, n_epochs=30, lr_all=0.01, reg_all=0.1, random_state=42)
svd_model.fit(trainset)

print("SVD model trained.")

SVD model trained.


In [8]:
movies['genre_list'] = movies['genres'].str.split('|')
all_genres = sorted(set(g for genres in movies['genre_list'] for g in genres))
genre_matrix = pd.DataFrame(0, index=movies['movieId'], columns=all_genres)

for idx, row in movies.iterrows():
    for genre in row['genre_list']:
        genre_matrix.loc[row['movieId'], genre] = 1

movie_popularity = train_df.groupby('movieId').size().reset_index(name='rating_count')

print("Genre matrix:", genre_matrix.shape)

Genre matrix: (9742, 20)


In [9]:
train_avg_rating = train_df.groupby('movieId')['rating'].mean()
train_rating_count = train_df.groupby('movieId')['rating'].count()

train_movie_stats = pd.DataFrame({
    'avg_rating': train_avg_rating,
    'rating_count': train_rating_count
}).reset_index()

C_train = train_df['rating'].mean()
m_train = train_movie_stats['rating_count'].quantile(0.90)

train_movie_stats['weighted_rating'] = (
    (train_movie_stats['rating_count'] / (train_movie_stats['rating_count'] + m_train)) * train_movie_stats['avg_rating']
    + (m_train / (train_movie_stats['rating_count'] + m_train)) * C_train
)

train_movie_stats = train_movie_stats.merge(movies[['movieId', 'title']], on='movieId')

popularity_ranked = train_movie_stats.sort_values('weighted_rating', ascending=False)
print(popularity_ranked[['title', 'weighted_rating']].head(10))

                                                  title  weighted_rating
277                    Shawshank Redemption, The (1994)         4.336498
656                               Godfather, The (1972)         4.195942
224           Star Wars: Episode IV - A New Hope (1977)         4.176948
2207                                  Fight Club (1999)         4.166129
46                           Usual Suspects, The (1995)         4.156300
894   Raiders of the Lost Ark (Indiana Jones and the...         4.146392
600   Dr. Strangelove or: How I Learned to Stop Worr...         4.139715
459                             Schindler's List (1993)         4.138274
1924                                 Matrix, The (1999)         4.137110
893                          Princess Bride, The (1987)         4.132613


In [10]:
def recommend_tier1_svd(user_id, svd_model, movies, train_df, top_n=10):
    all_movie_ids = movies['movieId'].unique()
    already_rated = train_df[train_df['userId'] == user_id]['movieId'].tolist()
    candidates = [m for m in all_movie_ids if m not in already_rated]
    
    predictions = [(m, svd_model.predict(user_id, m).est) for m in candidates]
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    top_movie_ids = [m for m, score in predictions[:top_n]]
    return top_movie_ids

In [11]:
def recommend_tier2_content(liked_movie_ids, genre_matrix, movie_popularity, movies, top_n=10):
    liked_movie_ids = [m for m in liked_movie_ids if m in genre_matrix.index]
    
    if len(liked_movie_ids) == 0:
        return []
    
    profile = genre_matrix.loc[liked_movie_ids].mean(axis=0)
    sims = cosine_similarity([profile], genre_matrix)[0]
    
    sim_df = pd.DataFrame({
        'movieId': genre_matrix.index,
        'similarity': sims
    })
    
    sim_df = sim_df.merge(movie_popularity[['movieId', 'rating_count']], on='movieId', how='left')
    sim_df['rating_count'] = sim_df['rating_count'].fillna(0)
    
    # Exclude the movies they already picked as favorites
    sim_df = sim_df[~sim_df['movieId'].isin(liked_movie_ids)]
    
    sim_df = sim_df.sort_values(['similarity', 'rating_count'], ascending=[False, False])
    return sim_df['movieId'].head(top_n).tolist()

In [12]:
def recommend_tier3_popularity(popularity_ranked, top_n=10):
    return popularity_ranked['movieId'].head(top_n).tolist()

In [13]:
def hybrid_recommend(user_id, train_df, svd_model, genre_matrix, movie_popularity, 
                      popularity_ranked, movies, liked_movie_ids=None, top_n=10):
    
    user_has_history = user_id in train_df['userId'].values
    
    if user_has_history:
        print(f"User {user_id}: Tier 1 (SVD) — has training history")
        return recommend_tier1_svd(user_id, svd_model, movies, train_df, top_n)
    
    elif liked_movie_ids is not None and len(liked_movie_ids) > 0:
        print(f"User {user_id}: Tier 2 (Content-Based) — new user, provided preferences")
        return recommend_tier2_content(liked_movie_ids, genre_matrix, movie_popularity, movies, top_n)
    
    else:
        print(f"User {user_id}: Tier 3 (Popularity) — no history, no preferences given")
        return recommend_tier3_popularity(popularity_ranked, top_n)

In [14]:
# Tier 1 test: an existing user
existing_user = train_df['userId'].iloc[0]
recs_tier1 = hybrid_recommend(existing_user, train_df, svd_model, genre_matrix, movie_popularity, popularity_ranked, movies, top_n=5)
print(movies[movies['movieId'].isin(recs_tier1)][['title', 'genres']])

print("\n---\n")

# Tier 2 test: a "new" user who picked 5 favorite movies (simulate with fake ID + real movie choices)
fake_new_user_favs = movies[movies['title'].str.contains('Toy Story|Lion King|Aladdin', case=False, na=False)]['movieId'].tolist()
recs_tier2 = hybrid_recommend(999999, train_df, svd_model, genre_matrix, movie_popularity, popularity_ranked, movies, liked_movie_ids=fake_new_user_favs, top_n=5)
print(movies[movies['movieId'].isin(recs_tier2)][['title', 'genres']])

print("\n---\n")

# Tier 3 test: a user with zero info
recs_tier3 = hybrid_recommend(888888, train_df, svd_model, genre_matrix, movie_popularity, popularity_ranked, movies, top_n=5)
print(movies[movies['movieId'].isin(recs_tier3)][['title', 'genres']])

User 429: Tier 1 (SVD) — has training history
                                    title     genres
796                 Secrets & Lies (1996)      Drama
918                            Ran (1985)  Drama|War
2582  Guess Who's Coming to Dinner (1967)      Drama
4396       Trial, The (Procès, Le) (1962)      Drama
8466                      Whiplash (2014)      Drama

---

User 999999: Tier 2 (Content-Based) — new user, provided preferences
                                 title  \
1177                   Hercules (1997)   
1706                       Antz (1998)   
2287                 Robin Hood (1973)   
3000  Emperor's New Groove, The (2000)   
3568             Monsters, Inc. (2001)   

                                           genres  
1177  Adventure|Animation|Children|Comedy|Musical  
1706  Adventure|Animation|Children|Comedy|Fantasy  
2287  Adventure|Animation|Children|Comedy|Musical  
3000  Adventure|Animation|Children|Comedy|Fantasy  
3568  Adventure|Animation|Children|Comedy|Fantas

In [15]:
def hybrid_predict_rating(user_id, movie_id, train_df, svd_model, train_movie_stats, C_train):
    user_has_history = user_id in train_df['userId'].values
    
    if user_has_history:
        # Tier 1: SVD prediction
        return svd_model.predict(user_id, movie_id).est
    else:
        # Tier 3: fall back to movie's popularity weighted rating (or global mean if movie also unseen)
        row = train_movie_stats[train_movie_stats['movieId'] == movie_id]
        if len(row) > 0:
            return row['weighted_rating'].values[0]
        else:
            return C_train

In [16]:
test_df = test_df.copy()
test_df['hybrid_predicted'] = test_df.apply(
    lambda row: hybrid_predict_rating(row['userId'], row['movieId'], train_df, svd_model, train_movie_stats, C_train),
    axis=1
)

from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse_hybrid = np.sqrt(mean_squared_error(test_df['rating'], test_df['hybrid_predicted']))
mae_hybrid = mean_absolute_error(test_df['rating'], test_df['hybrid_predicted'])

print(f"Hybrid Model — RMSE: {rmse_hybrid:.4f}")
print(f"Hybrid Model — MAE: {mae_hybrid:.4f}")

Hybrid Model — RMSE: 1.0156
Hybrid Model — MAE: 0.7922


In [17]:
testset_full = list(zip(test_df['userId'], test_df['movieId'], test_df['rating']))
predictions_svd_only = svd_model.test(testset_full)

print("Pure SVD (no hybrid fallback):")
accuracy.rmse(predictions_svd_only)
accuracy.mae(predictions_svd_only)

Pure SVD (no hybrid fallback):
RMSE: 1.0052
MAE:  0.7772


np.float64(0.7771574331924118)

In [18]:
known_users_in_test = test_df['userId'].isin(train_df['userId'].unique()).sum()
total_test_rows = len(test_df)

print(f"Test rows with known users (Tier 1, SVD): {known_users_in_test} / {total_test_rows} ({known_users_in_test/total_test_rows:.2%})")
print(f"Test rows with unknown users (Tier 3, Popularity fallback): {total_test_rows - known_users_in_test} / {total_test_rows} ({(total_test_rows-known_users_in_test)/total_test_rows:.2%})")

Test rows with known users (Tier 1, SVD): 1921 / 20168 (9.52%)
Test rows with unknown users (Tier 3, Popularity fallback): 18247 / 20168 (90.48%)
